In [ ]:
!pip install torch torchaudio librosa soundfile audioread

In [4]:
import torch
import torchaudio
from datasets import load_dataset, Audio
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import re

In [5]:
DATASET_PATH = r"C:\Users\user\pytorch-speech-recognition\sberdevices_golos_100h_farfield"
MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-russian"
NUM_SAMPLES = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^а-яё0-9 ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def wer(ref, hyp):
    r = ref.split()
    h = hyp.split()

    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]

    for i in range(len(r) + 1):
        d[i][0] = i
    for j in range(len(h) + 1):
        d[0][j] = j

    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            if r[i - 1] == h[j - 1]:
                cost = 0
            else:
                cost = 1

            d[i][j] = min(
                d[i - 1][j] + 1,      # deletion
                d[i][j - 1] + 1,      # insertion
                d[i - 1][j - 1] + cost  # substitution
            )

    return d[len(r)][len(h)] / max(1, len(r))

In [7]:
print("[1] Loading dataset...")

dataset = load_dataset(
    "parquet",
    data_dir=DATASET_PATH,
    split="train"
)

dataset = dataset.cast_column("audio", Audio())

[1] Loading dataset...


Generating train split: 9570 examples [00:04, 1916.59 examples/s]
Generating validation split: 933 examples [00:01, 833.50 examples/s]
Generating test split: 1916 examples [00:01, 1287.93 examples/s]


In [8]:
print("[2] Loading model...")

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

[2] Loading model...


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--jonatasgrosman--wav2vec2-large-xlsr-53-russian. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 424/424 [00:00<00:00, 5106.40it/s]


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [12]:
print("[3] Running test...")

total_wer = 0

for i in range(NUM_SAMPLES):
    sample = dataset[i]

    # audio
    audio = sample["audio"]
    waveform = torch.tensor(audio["array"]).unsqueeze(0)
    sr = audio["sampling_rate"]

    # resample
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)

    # mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # model input
    input_values = processor(
        waveform.squeeze(),
        return_tensors="pt",
        sampling_rate=16000
    ).input_values.to(DEVICE)

    # inference
    with torch.no_grad():
        logits = model(input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.decode(pred_ids[0])

    # clean
    gt = (
        sample.get("text") or
        sample.get("transcription") or
        sample.get("sentence") or
        sample.get("normalized_text")
    )
    pred = clean_text(pred_text)

    sample_wer = wer(gt, pred)
    total_wer += sample_wer

    print(f"\n--- SAMPLE {i} ---")
    print("GT   :", gt)
    print("PRED :", pred)
    print("WER  :", round(sample_wer, 3))

[3] Running test...

--- SAMPLE 0 ---
GT   : джой источники истории турции
PRED : джоуиспесеньтие история турция
WER  : 1.0

--- SAMPLE 1 ---
GT   : сбер когда начался московская паника тысяча девятьсот сорок первого года
PRED : сберь когда начался московская паника тыше девятьсот сорук первого года
WER  : 0.3

--- SAMPLE 2 ---
GT   : сколько денег осталось на сбербанке
PRED : холька денег осталось на сбербанкин
WER  : 0.4

--- SAMPLE 3 ---
GT   : афина выруби освещение
PRED : афейна вырвоби освущенья
WER  : 1.0

--- SAMPLE 4 ---
GT   : сбер какой сегодня день недели
PRED : сбер покой сегодня день неделе
WER  : 0.4

--- SAMPLE 5 ---
GT   : сбер закинь на мобилу три сотки
PRED : в берзакеном обелутя сотки
WER  : 0.833

--- SAMPLE 6 ---
GT   : афина октябрьская революция в честь кого назвали
PRED : афина актябрьский революции в чесь кого назвали
WER  : 0.429

--- SAMPLE 7 ---
GT   : афина пополни баланс моего мобильного пятьсот рублей
PRED : афина по помню балансного мобильного питсот ру

In [13]:
avg_wer = total_wer / NUM_SAMPLES

print("\n======================")
print("FINAL WER:", round(avg_wer, 3))
print("======================") 


FINAL WER: 0.646
